# PathSnap: End-to-End Map-Matching & Sequential Route Reconstruction
**Team: PathSnap_693_694_697_698 | PES University**

This interactive walkthrough demonstrates the complete pipeline:
1. **Raw Dataset Exploration & Preprocessing** (80 training trips, 10 testing trips)
2. **Vectorized Spatial Candidate Generation & 15-Feature Engineering**
3. **Multi-Model Benchmark Suite** (Logistic Regression, KNN, Random Forest, XGBoost, LightGBM, MLP)
4. **Bayesian Hyperparameter Optimization** (`skopt.BayesSearchCV` tuning LightGBM)
5. **HMM & Log-Space Viterbi Sequential Route Decoding**
6. **Interactive Folium Geographical Visualisation**

In [1]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium

# Add project src to path
sys.path.insert(0, os.path.abspath('../src'))
from data_processing import load_trip, build_adjacency, generate_trip_features
from hmm_viterbi import HMMRouteReconstructor
from model_training import FEATURE_COLS
print('Environment initialized successfully.')

## 1. Exploratory Data Analysis & Partition Statistics
Inspecting dataset summary files across the 80 training trips and 10 testing trips.

In [2]:
train_summary = pd.read_csv('../results/train_summary.csv')
test_summary = pd.read_csv('../results/test_summary.csv')

print('=== TRAINING PARTITION (80 Trips) ===')
print(f'Total GPS Observations: {train_summary["n_observations"].sum():,}')
print(f'Average Observations/Trip: {train_summary["n_observations"].mean():.1f}')
print(f'Total Trajectory Kilometers: {train_summary["trajectory_length_m"].sum() / 1000.0:.1f} km')
print(f'Total Road Graph Nodes: {train_summary["n_nodes"].sum():,}')
print(f'Total Road Graph Arcs: {train_summary["n_arcs"].sum():,}')

print('\n=== STUDENT TESTING PARTITION (10 Trips) ===')
print(f'Total GPS Observations: {test_summary["n_observations"].sum():,}')
print(f'Average Observations/Trip: {test_summary["n_observations"].mean():.1f}')
print(f'Total Trajectory Kilometers: {test_summary["trajectory_length_m"].sum() / 1000.0:.1f} km')

## 2. Inspecting a Raw Trip File Structure
Loading `.track`, `.nodes`, `.arcs`, and reference `.route` for student-testing trip `00000000`.

In [3]:
track, nodes, arcs, route = load_trip('../Student folder/student_test', '00000000')
print(f'Trip 00000000 GPS Observations: {len(track)}')
print(f'Road Network Nodes: {len(nodes)}')
print(f'Road Network Arcs: {len(arcs)}')
print(f'Reference Route Segment Count: {len(route) if route is not None else 0}')

display(track.head(5))

## 3. Spatial Candidate Generation & 15-Feature Engineering
Using the 2D Spatial Hash Grid index to generate candidate road segments within 50m search radius.

In [4]:
features_df, meta = generate_trip_features('../Student folder/student_test', '00000000', max_candidates=15)
print(f'Extracted {len(features_df):,} candidate pairs across {len(meta["track"])} GPS observations')
print(f'Engineered Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
display(features_df[['obs_idx', 'arc_id', 'distance', 'bearing_diff', 'gps_speed', 'candidate_rank', 'cos_bearing_diff', 'label']].head(5))

## 4. Multi-Model Benchmark Suite & Comparative Results
Inspecting the recorded metrics across the 6 benchmarked algorithm families.

In [5]:
model_comp = pd.read_csv('../results/model_comparison.csv')
display(model_comp[['model', 'family', 'cv_f1_macro', 'test_f1_macro', 'top1_accuracy', 'route_continuity', 'test_log_loss', 'inference_time']])

## 5. Bayesian Hyperparameter Optimization Results
Inspecting the best parameters found by `skopt.BayesSearchCV` for LightGBM and the resulting performance uplift.

In [6]:
with open('../results/bayesian_opt_results.json') as f:
    bayes_res = json.load(f)
print('Best Hyperparameters found by Bayesian Optimization:')
for k, v in bayes_res['best_params'].items():
    print(f'  {k}: {v}')

with open('../results/tuned_vs_untuned.json') as f:
    t_vs_u = json.load(f)
display(pd.DataFrame(t_vs_u).T[['cv_f1_macro', 'top1_accuracy', 'test_f1_macro', 'test_log_loss', 'route_continuity']])

## 6. HMM & Log-Space Viterbi Route Reconstruction
Running the sequential decoder and comparing classifier-only vs classifier+HMM predictions.

In [7]:
with open('../results/hmm_comparison.json') as f:
    hmm_comp = json.load(f)

print('=== HMM / VITERBI RECONSTRUCTION UPLIFT ===')
print(f'Classifier-Only Pointwise Accuracy: {hmm_comp["classifier_only"]["avg_pointwise_accuracy"]*100:.2f}%')
print(f'Classifier + HMM Pointwise Accuracy:  {hmm_comp["hmm_viterbi"]["avg_pointwise_accuracy"]*100:.2f}% (Uplift: +{hmm_comp["improvements"]["accuracy_gain"]*100:.2f}%)')
print(f'Classifier-Only Route Continuity:   {hmm_comp["classifier_only"]["avg_route_continuity"]*100:.2f}%')
print(f'Classifier + HMM Route Continuity:    {hmm_comp["hmm_viterbi"]["avg_route_continuity"]*100:.2f}% (Uplift: +{hmm_comp["improvements"]["continuity_gain"]*100:.2f}%)')

per_trip_df = pd.DataFrame(hmm_comp['per_trip_details'])
display(per_trip_df[['trip_id', 'n_observations', 'classifier_accuracy', 'hmm_accuracy', 'accuracy_delta', 'classifier_continuity', 'hmm_continuity']])

## 7. Interactive Geographical Visualisation
The interactive Folium maps generated by PathSnap can be inspected directly in the browser.
The maps visualize:
- **Raw GPS Points** (Blue circles)
- **Road Network Arcs** (Gray lines)
- **Ground Truth Route** (Dashed green line)
- **PathSnap Predicted Route** (Solid red line)

In [8]:
map_path = '../visualizations/maps/test_trip_00000000_map.html'
if os.path.exists(map_path):
    print(f'Interactive HTML Map available at: {os.path.abspath(map_path)}')
    print('Open in browser or inspect the embedded iframe.')
else:
    print('Generate maps by running src/visualizations.py')